# Titanic · DNN

Исследуем MLP с one-hot и EmbeddingMLP. Основная метрика — Accuracy, дополнительная — ROC-AUC; checkpoint выбираем по обычной validation BCE.

**Запуск текущего эксперимента:** ячейки сверху вниз с настройками config.yaml. Графики относятся к этому запуску; сохранённых выводов исполняемых ячеек сейчас нет. Исторические результаты и окончательный выбор описаны в конце.

Обучение и оценка — в [src/dnn_training.py](../src/dnn_training.py). dnn.device выбирает cpu, cuda или auto; явный cuda требует доступной CUDA. Полная внешняя CV вынесена в отдельный скрипт, ссылка — в итоговом разделе.


In [ ]:
import os
import sys
from pathlib import Path

# Ядро может стартовать из корня проекта или из notebooks.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config.yaml").is_file() or not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Откройте ноутбук из корня Titanic-ML или папки notebooks")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # Пути в конфиге и ячейках заданы от корня проекта.

import copy
import hashlib
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from src.dnn_training import evaluate, train_model, create_mlp, EmbeddingMLP, resolve_device
from src.config import load_config

%load_ext autoreload
%autoreload 2

config = load_config()
dnn_config = config["dnn"]
device = resolve_device(dnn_config.get("device", "cpu"))
print(f"DNN device: {device}")
SEED = config["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

run_config = {
    "seed": SEED,
    "train_path": config["data"]["train_path"],
    "target": config["data"]["target"],
    "dnn": copy.deepcopy(dnn_config),
}
config_json = json.dumps(run_config, ensure_ascii=False, sort_keys=True)
config_id = hashlib.sha256(config_json.encode("utf-8")).hexdigest()[:12]

## Данные и обработка

Источник и target задаются в `data`, остальные настройки — в `dnn` общего `config.yaml`. Исходный baseline использует семь признаков, стратифицированное разбиение 80/20, медианы, StandardScaler и one-hot. Предобработка обучается только на train.

Таблица ниже показывает фактический размер выборок и долю выживших.

Для Embedding числовой блок имеет тип `float32`, категориальные индексы — `long`. Батч содержит `(numeric, categorical, y)`; для one-hot сохранён формат `(X, y)`. Размеры таблиц категорий находятся в `categorical_cardinalities`. Модель и тренировочные функции поддерживают оба формата. Маркер пропуска после `fillna` кодируется как известная категория; индекс 0 зарезервирован для неизвестных значений, индекс 1 остаётся резервным.

In [ ]:
numeric_features = dnn_config["features"]["numeric"]
categorical_features = dnn_config["features"]["categorical"]
preprocessing_config = dnn_config["preprocessing"]
split_config = dnn_config["split"]
train_path = config["data"]["train_path"]
train = pd.read_csv(train_path)

categorical_encoder_name = preprocessing_config["categorical_encoder"]
if categorical_encoder_name not in ("one-hot", "embedding"):
    raise ValueError("categorical_encoder должен быть 'one-hot' или 'embedding'")
is_one_hot_categorical_encoder = categorical_encoder_name == "one-hot"

X = train[categorical_features + numeric_features].copy()
y = train[config["data"]["target"]].copy()
X[categorical_features] = X[categorical_features].fillna(
    preprocessing_config["categorical_missing_value"],
)

train_X, val_X, train_y, val_y = train_test_split(
    X, y,
    test_size=split_config["validation_size"],
    random_state=split_config["random_state"],
    stratify=y if split_config["stratify"] else None,
)

numeric_steps = [SimpleImputer(strategy=preprocessing_config["numeric_imputer_strategy"])]
if preprocessing_config["scale_numeric"]:
    numeric_steps.append(StandardScaler())

if is_one_hot_categorical_encoder:
    categorical_encoder = OneHotEncoder(
        sparse_output=False,
        handle_unknown=preprocessing_config["handle_unknown"],
    )
else:
    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        encoded_missing_value=-1,
        unknown_value=-2,
    )

preprocessor = ColumnTransformer([
    ("numeric", make_pipeline(*numeric_steps), numeric_features),
    ("categorical", categorical_encoder, categorical_features),
])
transformed_train_X = preprocessor.fit_transform(train_X)
transformed_val_X = preprocessor.transform(val_X)

assert train_X.index.intersection(val_X.index).empty
assert train_X.index.equals(train_y.index) and val_X.index.equals(val_y.index)
assert np.isfinite(transformed_train_X).all() and np.isfinite(transformed_val_X).all()
assert transformed_train_X.shape[1] == transformed_val_X.shape[1]

data_summary = pd.DataFrame({
    "Пассажиров": [len(train_X), len(val_X)],
    "Признаков после обработки": [transformed_train_X.shape[1], transformed_val_X.shape[1]],
    "Доля выживших": [train_y.mean(), val_y.mean()],
}, index=["Train", "Validation"])
display(data_summary.round(4))

In [ ]:
tensor_train_y = torch.tensor(train_y.to_numpy(), dtype=torch.float32).reshape(-1, 1)
tensor_val_y = torch.tensor(val_y.to_numpy(), dtype=torch.float32).reshape(-1, 1)

if is_one_hot_categorical_encoder:
    # One-hot: все признаки передаются одним float32-тензором.
    tensor_train_X = torch.tensor(transformed_train_X, dtype=torch.float32)
    tensor_val_X = torch.tensor(transformed_val_X, dtype=torch.float32)
    train_dataset = TensorDataset(tensor_train_X, tensor_train_y)
    val_dataset = TensorDataset(tensor_val_X, tensor_val_y)
else:
    numeric_slice = preprocessor.output_indices_["numeric"]
    categorical_slice = preprocessor.output_indices_["categorical"]

    tensor_train_numeric = torch.tensor(
        transformed_train_X[:, numeric_slice], dtype=torch.float32,
    )
    tensor_val_numeric = torch.tensor(
        transformed_val_X[:, numeric_slice], dtype=torch.float32,
    )

    # Сдвиг только категорий: unknown -2 -> 0, missing -1 -> 1.
    # После fillna маркер пропуска является обычной известной категорией.
    train_category_indices = transformed_train_X[:, categorical_slice] + 2
    val_category_indices = transformed_val_X[:, categorical_slice] + 2
    assert np.equal(train_category_indices, np.floor(train_category_indices)).all()
    assert np.equal(val_category_indices, np.floor(val_category_indices)).all()
    tensor_train_categorical = torch.tensor(train_category_indices, dtype=torch.long)
    tensor_val_categorical = torch.tensor(val_category_indices, dtype=torch.long)

    # Число строк будущей Embedding-таблицы для каждого признака.
    ordinal_encoder = preprocessor.named_transformers_["categorical"]
    categorical_cardinalities = [len(categories) + 2 for categories in ordinal_encoder.categories_]
    for column, cardinality in enumerate(categorical_cardinalities):
        for indices in (tensor_train_categorical, tensor_val_categorical):
            assert ((indices[:, column] >= 0) & (indices[:, column] < cardinality)).all()

    train_dataset = TensorDataset(
        tensor_train_numeric, tensor_train_categorical, tensor_train_y,
    )
    val_dataset = TensorDataset(
        tensor_val_numeric, tensor_val_categorical, tensor_val_y,
    )

loader_config = dnn_config["loader"]
train_loader = DataLoader(
    train_dataset,
    batch_size=loader_config["batch_size"],
    shuffle=loader_config["train_shuffle"],
    num_workers=loader_config["num_workers"],
)
val_loader = DataLoader(
    val_dataset,
    batch_size=loader_config["batch_size"],
    shuffle=loader_config["validation_shuffle"],
    num_workers=loader_config["num_workers"],
)

print("Train tensors:", [(tuple(t.shape), str(t.dtype)) for t in train_dataset.tensors])
print("Validation tensors:", [(tuple(t.shape), str(t.dtype)) for t in val_dataset.tensors])


## MLP и обучение

Режим `dnn.preprocessing.categorical_encoder` выбирает `one-hot` или `embedding`. Для one-hot создаётся MLP; для Embedding — отдельная таблица на каждый признак и та же полносвязная часть. Размер векторов задаёт `dnn.model.embedding_dim`.

Seed сбрасывается перед созданием выбранной модели. Число числовых признаков и размеры таблиц берутся из обработанных train-данных. `input_features` — ширина результата sklearn, `mlp_input_features` — фактический вход MLP (для текущих настроек 13 и 10 соответственно режиму).

Criterion обучения может включать `pos_weight`, но checkpoint и early stopping всегда оцениваются по обычной BCE. `train_model` восстанавливает лучшие веса. `predict` возвращает логиты; для loader без меток указываем `has_targets=False`.


In [ ]:
in_features = transformed_train_X.shape[1]
model_config = dnn_config["model"]
training_config = dnn_config["training"]
optimizer_config = dnn_config["optimizer"]
scheduler_config = dnn_config["scheduler"]
threshold = dnn_config["evaluation"]["threshold"]

# Новый запуск начинается с одинакового состояния генераторов случайных чисел.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if is_one_hot_categorical_encoder:
    model = create_mlp(model_config, in_features)
    mlp_input_features = in_features
else:
    numeric_features_count = tensor_train_numeric.shape[1]
    model = EmbeddingMLP(
        model_config, numeric_features_count, categorical_cardinalities,
        embedding_dim=model_config["embedding_dim"],
    )
    mlp_input_features = model.mlp[0].in_features

# Переносим модель до создания optimizer; Dataset остаётся на CPU.
model = model.to(device)

if training_config["loss_fn"] != "BCEWithLogitsLoss":
    raise ValueError("Текущий бинарный цикл рассчитан на BCEWithLogitsLoss")
loss_fn_params = {}
pos_weight = training_config.get("loss_fn_params", {}).get("pos_weight")
if pos_weight is not None:
    loss_fn_params["pos_weight"] = torch.tensor(
        [pos_weight], dtype=torch.float32, device=next(model.parameters()).device,
    )
criterion = nn.BCEWithLogitsLoss(**loss_fn_params)
eval_criterion = nn.BCEWithLogitsLoss()
optimizer = getattr(torch.optim, optimizer_config["name"])(model.parameters(), **optimizer_config["params"])
scheduler = None 
if scheduler_config["name"] is not None:
    scheduler = getattr(torch.optim.lr_scheduler, scheduler_config["name"])(
        optimizer, **scheduler_config["params"],
    )

(
    train_loss_history,
    validation_loss_history,
    validation_accuracy_history,
    validation_roc_auc_score,
    lr_history,
    best_model,
) = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    eval_criterion=eval_criterion,
    optimizer=optimizer,
    epoch=training_config["max_epochs"],
    min_epoch=training_config["min_epochs"],
    patience=training_config["patience"],
    min_delta=training_config["min_delta"],
    verbose=training_config["verbose"],
    threshold=threshold,
    scheduler=scheduler,
)

best_validation_metrics = evaluate(model, val_loader, eval_criterion, threshold=threshold)
assert np.isclose(best_validation_metrics["loss"], best_model["loss"])
print(f"Выполнено эпох: {len(train_loss_history)} · Лучшая эпоха: {best_model['epoch']}")
display(pd.DataFrame([best_validation_metrics]).rename(columns={
    "loss": "Validation loss", "accuracy": "Accuracy", "roc_auc": "ROC-AUC",
}).round(5))

## Кривые обучения

Train loss усредняется во время обновления весов; validation loss измеряется после эпохи. Пунктир отмечает лучший checkpoint по validation loss.

In [ ]:
epochs = range(1, len(train_loss_history) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4), layout="constrained")

axes[0].plot(epochs, train_loss_history, label="Train", color="#2563eb")
axes[0].plot(epochs, validation_loss_history, label="Validation", color="#ea580c")
axes[0].set(title="Ошибка обучения", ylabel="BCE loss")

axes[1].plot(epochs, validation_accuracy_history, label="Accuracy", color="#2563eb")
axes[1].plot(epochs, validation_roc_auc_score, label="ROC-AUC", color="#16a34a")
axes[1].set(
    title="Качество на validation",
    ylabel="Значение метрики",
    ylim=(0.5, 1),
)

axes[2].plot(epochs, lr_history, label="LR", color="#9333ea")
axes[2].set(title="Learning rate", ylabel="LR")
axes[2].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

for ax in axes:
    ax.axvline(
        best_model["epoch"],
        color="#64748b",
        linestyle="--",
        alpha=0.7,
        label="Лучшая эпоха",
    )
    ax.set_xlabel("Эпоха")
    ax.grid(alpha=0.2)
    ax.legend(frameon=False)

plt.show()

## Журнал экспериментов

Одна строка — один запуск: признаки, обработка, разбиение, архитектура, обучение и метрики восстановленного checkpoint. Полный снимок настроек хранится в config_json; путь задаётся в dnn.results_path.

Идентификатор включает seed и хеш настроек: повтор одинакового запуска обновляет свою строку. В таблице ниже виден краткий результат, полные параметры остаются в CSV. Исторические метрики не пересчитываются; отсутствующие новые поля старых строк остаются пустыми.


In [ ]:
results_path = Path(dnn_config["results_path"])
experiment_id = f"{dnn_config['experiment_name']}_seed_{SEED}_{config_id}"
scaling_description = " + StandardScaler" if preprocessing_config["scale_numeric"] else ""
experiment_result = {
    "experiment": experiment_id,
    "model": "MLP" if is_one_hot_categorical_encoder else type(model).__name__,
    "categorical_encoder": categorical_encoder_name,
    "embedding_dim": None if is_one_hot_categorical_encoder else model_config["embedding_dim"],
    "categorical_cardinalities": (
        None if is_one_hot_categorical_encoder else json.dumps(categorical_cardinalities)
    ),
    "hidden_layers": ", ".join(map(str, model_config["hidden_layers"])),
    "activation": model_config["activation"],
    "batchnorm": model_config["batch_norm"],
    "dropout": model_config["dropout"],
    "data": train_path,
    "target": config["data"]["target"],
    "numeric_features": ", ".join(numeric_features),
    "categorical_features": ", ".join(categorical_features),
    "preprocessing": (
        f"numeric: {preprocessing_config['numeric_imputer_strategy']}{scaling_description}; "
        f"categorical: {preprocessing_config['categorical_missing_value']} + "
        + ("one-hot" if is_one_hot_categorical_encoder else "ordinal indices (+2) -> embedding")
    ),
    "split": json.dumps(split_config, sort_keys=True),
    "train_rows": len(train_X),
    "validation_rows": len(val_X),
    "input_features": in_features,
    "mlp_input_features": mlp_input_features,
    "seed": SEED,
    "optimizer": type(optimizer).__name__,
    "lr": optimizer_config["params"]["lr"],
    "weight_decay": optimizer.param_groups[0]["weight_decay"],
    "scheduler": scheduler_config,
    "scheduler_params": str(scheduler_config["params"]),
    "batch_size": loader_config["batch_size"],
    "loss_fn": type(criterion).__name__,
    "eval_loss_fn": type(eval_criterion).__name__,
    "pos_weight": 1.0 if criterion.pos_weight is None else criterion.pos_weight.item(),
    "max_epochs": training_config["max_epochs"],
    "min_epochs": training_config["min_epochs"],
    "patience": training_config["patience"],
    "min_delta": training_config["min_delta"],
    "epochs_run": len(train_loss_history),
    "best_epoch": best_model["epoch"],
    "parameters": sum(parameter.numel() for parameter in model.parameters()),
    "threshold": threshold,
    "device": str(next(model.parameters()).device),
    "torch_version": str(torch.__version__),
    "config_json": config_json,
    "val_loss": best_validation_metrics["loss"],
    "val_accuracy": best_validation_metrics["accuracy"],
    "val_roc_auc": best_validation_metrics["roc_auc"],
}

# Старые значения читаем как строки, чтобы не округлить прежние метрики при записи.
experiment_results = (
    pd.read_csv(results_path, dtype=object, keep_default_na=False)
    if results_path.exists() else pd.DataFrame()
)
if not experiment_results.empty:
    experiment_results = experiment_results.loc[experiment_results["experiment"] != experiment_id]
experiment_results = pd.concat([experiment_results, pd.DataFrame([experiment_result])], ignore_index=True)
results_path.parent.mkdir(parents=True, exist_ok=True)
experiment_results.to_csv(results_path, index=False)

summary_columns = [
    "experiment", "model", "categorical_encoder", "embedding_dim", "activation", "batchnorm", "dropout", "hidden_layers", "batch_size", "optimizer", "seed",
    "device", "epochs_run", "best_epoch", "val_loss", "val_accuracy", "val_roc_auc",
]
summary = experiment_results[summary_columns].copy()
for column in ("embedding_dim", "batch_size", "seed", "epochs_run", "best_epoch",
               "val_loss", "val_accuracy", "val_roc_auc"):
    summary[column] = pd.to_numeric(summary[column], errors="coerce")
display(summary.round(5))

## История: исходный baseline

При исходных настройках и seed 42 лучший checkpoint — эпоха 48, остановка — 73. После восстановления: **Accuracy 81.01%, ROC-AUC 0.86113, validation BCE 0.43190**. Это checkpoint по loss, а не максимальная Accuracy среди эпох.

После минимума validation loss обучение проявляло переобучение. Эти значения относятся к исходному baseline, а не текущему конфигу. Validation участвовала в выборе эпохи и параметров; напрямую сравнивать её метрики с внешней CV нельзя.


## История: архитектуры, активации и регуляризация

| Проверка | Наблюдение на validation, seed 42 |
|---|---|
| Ширина и глубина | [64]: Accuracy 82.12%; [8,32]: loss 0.42001, Accuracy 79.89%. Сложность не гарантировала выигрыша. |
| Опорная [24] | Accuracy 81.56%, loss 0.42504, ROC-AUC 0.86825. |
| Активации | ReLU лучше проверенных LeakyReLU, GELU, SiLU и Tanh для [24]. |
| BatchNorm | Без Dropout сохранил Accuracy, но ухудшил loss и ROC-AUC. |
| Dropout 0.05/0.10 | 147/179 правильных ответов против 146 без Dropout; loss и ROC-AUC ухудшились. |

Небольшой выигрыш Accuracy — чистый прирост одного правильного ответа, а не обязательно изменение прогноза только одного пассажира. Сильная регуляризация также не дала устойчивого выигрыша. Это промежуточная серия на одном разбиении; окончательная выбранная EmbeddingMLP описана ниже.


## История: scheduler

В контрольной паре с Adam и LR 0.006 косинусовый scheduler (T_max=50, eta_min=0.00003) дал Accuracy **81.01% против 81.56%** без него; loss 0.42318 против 0.42240. Повышение LR до 0.01 и 0.1 с расписанием также не улучшило лидера серии без scheduler (83.24%).

В проверенных настройках оставили постоянный LR. Вывод относится к одному seed и validation-разбиению; ошибочные запуски scheduler не являются проверкой его эффективности.


## Итог: внешняя CV и выбранная модель

Проверены две конфигурации на одинаковых пяти внешних стратифицированных фолдах (seed разбиения 42). Внутри каждого train выделена validation 20% для early stopping; обработка обучена на внутреннем train. Внешние пассажиры оцениваются только после восстановления checkpoint. Всего 10 обучений.

Общие настройки: ReLU, Adam, LR 0.003, batch 16, seed обучения 42, без BatchNorm, scheduler и весов классов.

| Конфигурация | Средняя Accuracy | Std по фолдам, п.п. | Средняя BCE | Средний ROC-AUC |
|---|---:|---:|---:|---:|
| One-hot [24], Dropout 0 | 81.82% | 1.55 | 0.42323 | 0.86670 |
| Embedding [24,32,16,8], dim 2, Dropout 0.15 | **83.05%** | 3.39 | **0.42165** | **0.86752** |

**Выбрана EmbeddingMLP [24,32,16,8], dim 2, Dropout 0.15.** Правило: максимальная средняя внешняя Accuracy; при равенстве — меньшая BCE, затем меньше параметров. Конфигурация сохранена в config.yaml; one-hot остаётся контролем.

Embedding выиграла два фолда, проиграла один, на двух Accuracy совпала. Чистый выигрыш — 11 правильных ответов (740/891 против 729/891), но разброс выше. Сравниваются полные конфигурации: результат нельзя приписать отдельно Embedding или глубине. Кандидаты ранее подбирались на этих данных; это исследовательская CV, не независимый тест.

**Код и повторение:** [scripts/dnn_cv.py](../scripts/dnn_cv.py), из корня `.venv\Scripts\python.exe -X utf8 scripts/dnn_cv.py`. Команда повторяет всю серию и обновляет отдельные CV-файлы, основной журнал не меняет.

**Подробности:** [метрики 10 запусков](../experiments/dnn_cv_results.csv), [сводка](../experiments/dnn_cv_summary.json), [разбиения](../experiments/dnn_cv_splits.json), [внешние прогнозы](../experiments/dnn_cv_predictions.csv). Пофолдовые значения, объединённый ROC-AUC и число параметров доступны в сохранённых результатах.
